# AprioriFIM: khai thác tập phổ biến và luật kết hợp (notebook mẫu)

Notebook mẫu cho Chương 3: cài gói `AprioriFIM` của nhóm, khai thác tập phổ biến và sinh luật kết hợp
(support, confidence, lift). Mọi kết quả đều do cài đặt Apriori của nhóm tính, không dùng thư viện FIM có sẵn.

**Chạy trên Colab:** *Runtime → Change runtime type → Julia*, rồi *Runtime → Run all*.
Mã nguồn: https://github.com/TaVanDuc2005/FIM_Group14

## 1. Cài đặt

In [1]:
# Tìm thư mục repo: mở notebook ngay trong repo (thư mục gốc hoặc notebooks/) thì dùng luôn;
# trên Colab thì shallow clone về. Gói chỉ dùng thư viện chuẩn nên không cần tải registry của Pkg.
is_repo(dir) = isfile(joinpath(dir, "src", "AprioriFIM.jl"))
const REPO = abspath(is_repo(".") ? "." : is_repo("..") ? ".." : "FIM_Group14")
is_repo(REPO) || run(`git clone --depth 1 https://github.com/TaVanDuc2005/FIM_Group14`)
using Pkg
Pkg.activate(REPO)
using AprioriFIM

Cloning into 'FIM_Group14'...
  Activating project at `/content/FIM_Group14`
[ Info: Precompiling AprioriFIM [5f544ea2-1aa2-4c25-b8c8-b024a6782403]
[ Info: Precompiling AprioriFIM [5f544ea2-1aa2-4c25-b8c8-b024a6782403] 

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up


## 2. Ví dụ nhỏ: giỏ hàng có tên sản phẩm

`encode_transactions` đổi tên item sang mã số nguyên (thuật toán làm việc trên số nguyên) và trả về
`labels` để đổi ngược lại. `minsup` dạng số thực là tỉ lệ (`0.4` = 40%), dạng số nguyên là số giao dịch.

In [2]:
baskets = [
    ["bread", "milk"],
    ["bread", "diapers", "beer", "eggs"],
    ["milk", "diapers", "beer", "cola"],
    ["bread", "milk", "diapers", "beer"],
    ["bread", "milk", "diapers", "cola"],
]
tx, labels = encode_transactions(baskets)
itemsets = apriori_bitarray(tx, 0.4)          # minsup 40% = 2 giao dịch

for (itemset, sup) in sort(collect(itemsets); by = p -> (-p[2], length(p[1]), p[1]))
    println(join(decode(itemset, labels), ", "), "   #SUP: ", sup)
end

bread   #SUP: 4
diapers   #SUP: 4
milk   #SUP: 4
beer   #SUP: 3
beer, diapers   #SUP: 3
bread, diapers   #SUP: 3
bread, milk   #SUP: 3
diapers, milk   #SUP: 3
cola   #SUP: 2
beer, bread   #SUP: 2
beer, milk   #SUP: 2
cola, diapers   #SUP: 2
cola, milk   #SUP: 2
beer, bread, diapers   #SUP: 2
beer, diapers, milk   #SUP: 2
bread, diapers, milk   #SUP: 2
cola, diapers, milk   #SUP: 2


## 3. Luật kết hợp, top theo lift

`association_rules(itemsets, số_giao_dịch; minconf, minlift = 0)` sinh mọi luật X ⇒ Y đạt ngưỡng
(thủ tục ap-genrules, mục 3 của Agrawal & Srikant 1994). Lift > 1: X và Y đi cùng nhau nhiều hơn ngẫu nhiên.

In [3]:
rules = association_rules(itemsets, length(tx); minconf = 0.6)
top = first(sort(rules; by = r -> (-r.lift, -r.confidence)), 10)
foreach(r -> println(format_rule(r, labels)), top)

{cola} => {diapers, milk}  sup = 2, conf = 1.0, lift = 1.667
{diapers, milk} => {cola}  sup = 2, conf = 0.667, lift = 1.667
{beer, milk} => {diapers}  sup = 2, conf = 1.0, lift = 1.25
{cola, milk} => {diapers}  sup = 2, conf = 1.0, lift = 1.25
{cola, diapers} => {milk}  sup = 2, conf = 1.0, lift = 1.25
{cola} => {milk}  sup = 2, conf = 1.0, lift = 1.25
{beer, bread} => {diapers}  sup = 2, conf = 1.0, lift = 1.25
{beer} => {diapers}  sup = 3, conf = 1.0, lift = 1.25
{cola} => {diapers}  sup = 2, conf = 1.0, lift = 1.25
{diapers} => {beer}  sup = 3, conf = 0.75, lift = 1.25


## 4. Dữ liệu dạng file

File định dạng SPMF (mỗi dòng một giao dịch, item là số nguyên cách nhau bởi dấu cách) đọc bằng
`read_transactions`. Kết quả ghi ra file bằng `write_itemsets` / `write_rules` (định dạng SPMF).

In [4]:
path = download("https://raw.githubusercontent.com/TaVanDuc2005/FIM_Group14/main/test/data/db/quest_medium.txt")
tx = read_transactions(path)
println(length(tx), " giao dịch")

@time itemsets = apriori_bitarray(tx, 0.02)
@time rules = association_rules(itemsets, length(tx); minconf = 0.5, minlift = 1.0)
println(length(itemsets), " tập phổ biến, ", length(rules), " luật")
foreach(println, first(sort(rules; by = r -> -r.lift), 5))

write_rules("rules.txt", rules)

3000 giao dịch
  0.016564 seconds (64.38 k allocations: 2.781 MiB)
  0.014057 seconds (44.00 k allocations: 1.991 MiB, 77.21% compilation time)
920 tập phổ biến, 1873 luật
{31, 37} => {13} (sup = 62, conf = 0.775, lift = 18.6)
{31, 116} => {13} (sup = 60, conf = 0.7692, lift = 18.4615)
{31, 59} => {13} (sup = 70, conf = 0.7216, lift = 17.3196)
{13} => {31, 59} (sup = 70, conf = 0.56, lift = 17.3196)
{13} => {37, 116} (sup = 65, conf = 0.52, lift = 15.9184)


"rules.txt"

## 5. Khung cho ứng dụng

Dữ liệu thật thường có item là chuỗi. File mà mỗi dòng là một giao dịch, item cách nhau bởi dấu phẩy
(ví dụ Groceries dạng basket) đọc bằng `read_labeled_transactions(path; delim = ',')`.
Dữ liệu dạng khác (bảng dài, log, chuỗi gen, visual word, ...) chỉ cần đưa về `Vector{Vector{T}}`,
mỗi phần tử là một giao dịch, rồi gọi `encode_transactions`.

```julia
raw = read_labeled_transactions("groceries.csv")     # Vector{Vector{String}}
tx, labels = encode_transactions(raw)
itemsets = apriori_bitarray(tx, 0.01)
rules = association_rules(itemsets, length(tx); minconf = 0.3)
for r in first(sort(rules; by = r -> -r.lift), 10)
    println(format_rule(r, labels))
end
```

**Lưu ý về bộ nhớ:** `apriori_bitarray` giữ một dãy bit dài bằng số giao dịch cho mỗi tập phổ biến.
Với dữ liệu dày (giao dịch dài, ít item) và minsup thấp, số tập phổ biến bùng nổ rất nhanh.
Nên bắt đầu từ minsup cao rồi giảm dần. `apriori` (bản cơ bản) tốn ít bộ nhớ hơn nhưng chậm hơn nhiều.